In [28]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel




In [29]:
df = pd.read_csv("../data/clean_hr_data.csv")
pd.set_option("display.max_colwidth", None)
df.head()

,id,job_title,location,connection,relevance,clean_job_title
0,1,2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional,"Houston, Texas",85,1,2019 c.t. bauer college business graduate magna cum laude aspire human resources professional houston texas 85
1,2,Native English Teacher at EPIK (English Program in Korea),Kanada,500,0,native english teacher epik english program korea kanada 500
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,1,aspiring human resources professional raleigh durham north carolina area 44
3,4,People Development Coordinator at Ryan,"Denton, Texas",500,0,people development coordinator ryan denton texas 500
4,5,Advisory Board Member at Celal Bayar University,"Izmir, Turkiye",500,0,advisory board member celal bayar university izmir turkiye 500


In [30]:
measure_rank_1 = "aspiring human resources" 
measure_rank_2 =  "seeking human resources"

BoW  is defined as how many times does the word appear?
TF-IDF is defined as how important/distinctive is this word compared with the rest of the dataset?

In [31]:
# BOW method 
# You fit and transform on the column
# you only transform the y1 and y2
vectorizer_bow = CountVectorizer()
X = vectorizer_bow.fit_transform(df['clean_job_title'])
y1 = vectorizer_bow.transform([measure_rank_1])
y2 = vectorizer_bow.transform([measure_rank_2])

In [32]:
score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)

In [33]:
df["similarity_1_bow"] = score_1.flatten()
df[df["similarity_1_bow"]>0.4][["clean_job_title", "similarity_1_bow"]].sort_values(
    by="similarity_1_bow",
    ascending=False
).head(10)

,clean_job_title,similarity_1_bow
21,aspiring human resources manager seek internship human resources houston texas area 7,0.745356
47,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri 103,0.615457
5,aspiring human resources specialist greater new york city area 1,0.577350
6,student humber college aspiring human resources generalist kanada 61,0.577350
2,aspiring human resources professional raleigh durham north carolina area 44,0.547723
20,business management major aspiring human resources manager monroe louisiana area 5,0.547723
12,aspiring human resources management student seek internship houston texas area 500,0.522233
30,aspiring human resources professional | energetic team focus leader austin texas area 174,0.500000
13,seek human resources opportunities chicago illinois 390,0.436436
36,human resources management major milpitas california 18,0.436436


In [34]:
df["similarity_2_bow"] = score_2.flatten()
df[df["similarity_2_bow"]>0.4][["clean_job_title", "similarity_2_bow"]].sort_values(
    by="similarity_2_bow",
    ascending=False
).head(10)

,clean_job_title,similarity_2_bow
21,aspiring human resources manager seek internship human resources houston texas area 7,0.730297
47,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri 103,0.603023
36,human resources management major milpitas california 18,0.534522
22,human resources professional greater boston area 16,0.534522
13,seek human resources opportunities chicago illinois 390,0.534522
37,director human resources ey greater atlanta area 349,0.500000
26,human resources generalist schwan amerika birlesik devletleri 500,0.500000
15,human resources staffing recruiting professional jackson mississippi area 500,0.471405
5,aspiring human resources specialist greater new york city area 1,0.471405
46,seek human resources position las vegas nevada area 48,0.471405


In [35]:
# Tfidf method
vectorizer_tfidf = TfidfVectorizer()
X = vectorizer_tfidf.fit_transform(df['clean_job_title'])
y1 = vectorizer_tfidf.transform([measure_rank_1])
y2 = vectorizer_tfidf.transform([measure_rank_2])

score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)

In [36]:
df["similarity_1_tfidf"] = score_1.flatten()
df[df["similarity_1_tfidf"]>0.3][["clean_job_title", "similarity_1_tfidf"]].sort_values(
    by="similarity_1_tfidf",
    ascending=False
).head(10)

,clean_job_title,similarity_1_tfidf
21,aspiring human resources manager seek internship human resources houston texas area 7,0.547299
5,aspiring human resources specialist greater new york city area 1,0.416505
12,aspiring human resources management student seek internship houston texas area 500,0.390410
2,aspiring human resources professional raleigh durham north carolina area 44,0.365241
20,business management major aspiring human resources manager monroe louisiana area 5,0.361899
6,student humber college aspiring human resources generalist kanada 61,0.345649
47,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri 103,0.308499


In [37]:
df["similarity_2_tfidf"] = score_2.flatten()
df[df["similarity_2_tfidf"]>0.3][["clean_job_title", "similarity_2_tfidf"]].sort_values(
    by="similarity_2_tfidf",
    ascending=False
).head(10)

,clean_job_title,similarity_2_tfidf
21,aspiring human resources manager seek internship human resources houston texas area 7,0.491453


In [38]:
#%pip install gensim

model = api.load('glove-wiki-gigaword-100')

Each word has a GloVe vector → average all word vectors → get one vector representing the entire job title.

In [39]:
# apply has the tendency to work on the rows automatically 
def get_title_vector(row):
    words = row.split()

    vectors = []

    for word in words:
        if word in model:
            vectors.append(model[word])

    return np.mean(vectors, axis=0)

In [40]:
df['vector_for_modeling'] = df["clean_job_title"].apply(get_title_vector)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)

In [41]:
X_glove = np.vstack(df["vector_for_modeling"])
# to make it work you need to use vstack function
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_glove
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_glove
)


In [42]:
df["similarity_1_glove"] = score_1.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_1_glove"]>0.3][["clean_job_title", "similarity_1_glove"]].sort_values(
    by="similarity_1_glove",
    ascending=False
).head(10)

,clean_job_title,similarity_1_glove
21,aspiring human resources manager seek internship human resources houston texas area 7,0.849718
24,aspiring human resources professional | passionate help create inclusive engage work environment new york new york 212,0.821569
13,seek human resources opportunities chicago illinois 390,0.811086
32,human resources professional world leader gis software highland california 50,0.809051
37,director human resources ey greater atlanta area 349,0.802835
12,aspiring human resources management student seek internship houston texas area 500,0.800376
8,seek human resources hris generalist positions greater philadelphia area 500,0.798086
22,human resources professional greater boston area 16,0.797593
5,aspiring human resources specialist greater new york city area 1,0.788184
30,aspiring human resources professional | energetic team focus leader austin texas area 174,0.784895


In [43]:
df["similarity_2_glove"] = score_2.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_2_glove"]>0.3][["clean_job_title", "similarity_2_glove"]].sort_values(
    by="similarity_2_glove",
    ascending=False
).head(10)




,clean_job_title,similarity_2_glove
13,seek human resources opportunities chicago illinois 390,0.874183
21,aspiring human resources manager seek internship human resources houston texas area 7,0.871159
8,seek human resources hris generalist positions greater philadelphia area 500,0.863006
42,seek human resources opportunity open travel relocation amerika birlesik devletleri 415,0.862108
37,director human resources ey greater atlanta area 349,0.826413
36,human resources management major milpitas california 18,0.825759
22,human resources professional greater boston area 16,0.820881
32,human resources professional world leader gis software highland california 50,0.809376
12,aspiring human resources management student seek internship houston texas area 500,0.808565
24,aspiring human resources professional | passionate help create inclusive engage work environment new york new york 212,0.801996


In [44]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

def get_title_vector_bert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )
# pt is py_torch, truncation for fitting what the model allow in size, padding makes the size the same
    with torch.no_grad():
        outputs = bert(**inputs)


    vector = outputs.last_hidden_state.mean(dim=1)

    return vector.squeeze().numpy()
# make the tensor a numpy array with one dimension 
df['bert_for_modeling'] = df["clean_job_title"].apply(get_title_vector_bert)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [45]:
X_bert = np.vstack(df["bert_for_modeling"])

y1_vector = get_title_vector_bert(measure_rank_1)
y2_vector = get_title_vector_bert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bert
)

df["similarity_1_bert"] = score_1.flatten()

df[df["similarity_1_bert"]>0.3][["clean_job_title", "similarity_1_bert"]].sort_values(
    by="similarity_1_bert",
    ascending=False
).head(10)


,clean_job_title,similarity_1_bert
13,seek human resources opportunities chicago illinois 390,0.749111
21,aspiring human resources manager seek internship human resources houston texas area 7,0.747322
30,aspiring human resources professional | energetic team focus leader austin texas area 174,0.735730
15,human resources staffing recruiting professional jackson mississippi area 500,0.720959
12,aspiring human resources management student seek internship houston texas area 500,0.720783
14,experienced retail manager aspire human resources professional austin texas area 57,0.719675
6,student humber college aspiring human resources generalist kanada 61,0.719633
32,human resources professional world leader gis software highland california 50,0.715453
51,director administration excellence logging katy texas 500,0.707068
5,aspiring human resources specialist greater new york city area 1,0.702381


In [46]:
df["similarity_2_bert"] = score_2.flatten()

df[df["similarity_2_bert"]>0.3][["clean_job_title", "similarity_2_bert"]].sort_values(
    by="similarity_2_bert",
    ascending=False
).head(10)

,clean_job_title,similarity_2_bert
13,seek human resources opportunities chicago illinois 390,0.801287
21,aspiring human resources manager seek internship human resources houston texas area 7,0.702566
15,human resources staffing recruiting professional jackson mississippi area 500,0.700950
8,seek human resources hris generalist positions greater philadelphia area 500,0.698048
46,seek human resources position las vegas nevada area 48,0.684404
40,seek employment opportunity customer service patient care torrance california 64,0.683962
51,director administration excellence logging katy texas 500,0.683022
3,people development coordinator ryan denton texas 500,0.679889
22,human resources professional greater boston area 16,0.679089
37,director human resources ey greater atlanta area 349,0.666376


In [47]:
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

sbert = AutoModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)


def get_title_vector_sbert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = sbert(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # Mean pooling, ignoring padding tokens
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["sbert_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_sbert
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [48]:
s_bert = np.vstack(df["sbert_for_modeling"])

y1_vector = get_title_vector_sbert(measure_rank_1)
y2_vector = get_title_vector_sbert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    s_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    s_bert
)

df["similarity_1_sbert"] = score_1.flatten()

df[df["similarity_1_sbert"]>0.3][["clean_job_title", "similarity_1_sbert"]].sort_values(
    by="similarity_1_sbert",
    ascending=False
).head(10)

,clean_job_title,similarity_1_sbert
2,aspiring human resources professional raleigh durham north carolina area 44,0.819170
5,aspiring human resources specialist greater new york city area 1,0.788344
47,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri 103,0.775312
21,aspiring human resources manager seek internship human resources houston texas area 7,0.731115
46,seek human resources position las vegas nevada area 48,0.710243
30,aspiring human resources professional | energetic team focus leader austin texas area 174,0.704329
6,student humber college aspiring human resources generalist kanada 61,0.702629
12,aspiring human resources management student seek internship houston texas area 500,0.701894
24,aspiring human resources professional | passionate help create inclusive engage work environment new york new york 212,0.699898
13,seek human resources opportunities chicago illinois 390,0.670924


In [49]:
df["similarity_2_sbert"] = score_2.flatten()

df[df["similarity_2_sbert"]>0.3][["clean_job_title", "similarity_2_sbert"]].sort_values(
    by="similarity_2_sbert",
    ascending=False
).head(10)

,clean_job_title,similarity_2_sbert
46,seek human resources position las vegas nevada area 48,0.761732
2,aspiring human resources professional raleigh durham north carolina area 44,0.734525
5,aspiring human resources specialist greater new york city area 1,0.717399
13,seek human resources opportunities chicago illinois 390,0.717316
8,seek human resources hris generalist positions greater philadelphia area 500,0.699338
48,human resources generalist loparex raleigh durham north carolina area 500,0.671056
47,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri 103,0.661074
21,aspiring human resources manager seek internship human resources houston texas area 7,0.658575
30,aspiring human resources professional | energetic team focus leader austin texas area 174,0.653204
24,aspiring human resources professional | passionate help create inclusive engage work environment new york new york 212,0.640741


In [50]:
tokenizer = AutoTokenizer.from_pretrained(
    "intfloat/e5-small-v2"
)

e5 = AutoModel.from_pretrained(
    "intfloat/e5-small-v2"
)


def get_title_vector_e5(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = e5(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["e5_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_e5
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [51]:
X_e5 = np.vstack(df["e5_for_modeling"])

y1_vector = get_title_vector_e5(measure_rank_1)
y2_vector = get_title_vector_e5(measure_rank_2)

In [52]:
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_e5
)

df["similarity_1_5e"] = score_1.flatten()

df[df["similarity_1_5e"]>0.3][["clean_job_title", "similarity_1_5e"]].sort_values(
    by="similarity_1_5e",
    ascending=False
).head(10)

,clean_job_title,similarity_1_5e
5,aspiring human resources specialist greater new york city area 1,0.900310
21,aspiring human resources manager seek internship human resources houston texas area 7,0.894750
47,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri 103,0.890414
2,aspiring human resources professional raleigh durham north carolina area 44,0.886353
12,aspiring human resources management student seek internship houston texas area 500,0.878086
24,aspiring human resources professional | passionate help create inclusive engage work environment new york new york 212,0.876692
30,aspiring human resources professional | energetic team focus leader austin texas area 174,0.864373
20,business management major aspiring human resources manager monroe louisiana area 5,0.863725
46,seek human resources position las vegas nevada area 48,0.863497
13,seek human resources opportunities chicago illinois 390,0.860379


In [53]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_e5
)

df["similarity_2_5e"] = score_2.flatten()

df[df["similarity_2_5e"]>0.3][["clean_job_title", "similarity_2_5e"]].sort_values(
    by="similarity_2_5e",
    ascending=False
).head(10)

,clean_job_title,similarity_2_5e
13,seek human resources opportunities chicago illinois 390,0.892398
46,seek human resources position las vegas nevada area 48,0.890624
21,aspiring human resources manager seek internship human resources houston texas area 7,0.882211
8,seek human resources hris generalist positions greater philadelphia area 500,0.876815
47,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri 103,0.872307
42,seek human resources opportunity open travel relocation amerika birlesik devletleri 415,0.872267
5,aspiring human resources specialist greater new york city area 1,0.865797
12,aspiring human resources management student seek internship houston texas area 500,0.865030
24,aspiring human resources professional | passionate help create inclusive engage work environment new york new york 212,0.860166
23,nortia staffing seek human resources payroll administrative professionals 408 709 2621 san jose california 500,0.859875


In [55]:
tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)

bge = AutoModel.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)


def get_title_vector_bge(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = bge(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy() 


df["bge_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_bge
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [56]:
X_bge = np.vstack(df["bge_for_modeling"])

y1_vector = get_title_vector_bge(measure_rank_1)
y2_vector = get_title_vector_bge(measure_rank_2)

In [57]:
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bge
)

df["similarity_1_bge"] = score_1.flatten()
# Chain the filtering and sorting together
df[df["similarity_1_bge"] > 0.3][["clean_job_title", "similarity_1_bge"]].sort_values(
    by="similarity_1_bge",
    ascending=False
).head(10)

,clean_job_title,similarity_1_bge
2,aspiring human resources professional raleigh durham north carolina area 44,0.786999
21,aspiring human resources manager seek internship human resources houston texas area 7,0.783562
24,aspiring human resources professional | passionate help create inclusive engage work environment new york new york 212,0.780927
5,aspiring human resources specialist greater new york city area 1,0.778894
12,aspiring human resources management student seek internship houston texas area 500,0.767548
47,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri 103,0.755488
13,seek human resources opportunities chicago illinois 390,0.754207
20,business management major aspiring human resources manager monroe louisiana area 5,0.747347
30,aspiring human resources professional | energetic team focus leader austin texas area 174,0.743157
6,student humber college aspiring human resources generalist kanada 61,0.731464


In [58]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bge
)

df["similarity_2_bge"] = score_2.flatten()

df[df["similarity_2_bge"] > 0.3][["clean_job_title", "similarity_2_bge"]].sort_values(
    by="similarity_2_bge",
    ascending=False
).head(10)

,clean_job_title,similarity_2_bge
13,seek human resources opportunities chicago illinois 390,0.828740
8,seek human resources hris generalist positions greater philadelphia area 500,0.774525
21,aspiring human resources manager seek internship human resources houston texas area 7,0.768068
46,seek human resources position las vegas nevada area 48,0.761926
2,aspiring human resources professional raleigh durham north carolina area 44,0.758372
42,seek human resources opportunity open travel relocation amerika birlesik devletleri 415,0.736612
5,aspiring human resources specialist greater new york city area 1,0.730112
24,aspiring human resources professional | passionate help create inclusive engage work environment new york new york 212,0.729630
15,human resources staffing recruiting professional jackson mississippi area 500,0.724346
23,nortia staffing seek human resources payroll administrative professionals 408 709 2621 san jose california 500,0.721670


Among the evaluated embedding approaches, E5 produced the most relevant job-title rankings. This is consistent with E5's retrieval-oriented training, which is specifically designed to represent semantically related texts close together in the embedding space.

In [59]:
def ranked_by_model(similarity_model, col, relevance):
    ranked = df.sort_values(similarity_model, ascending=False)

    return ranked[[col, similarity_model, relevance]]

In [60]:
def metrics(ranked, relevance):
    labels = ranked[relevance].to_numpy()
    positions = np.arange(1, len(labels) + 1)

    precision_5 = labels[:5].sum() / 5
    precision_10 = labels[:10].sum() / 10

    relevant_positions = positions[labels == 1]
    mrr = 1 / relevant_positions[0] if len(relevant_positions) else 0

    precision_at_each_position = np.cumsum(labels) / positions
    ap = (
        (precision_at_each_position * labels).sum() / labels.sum()
        if labels.sum() else 0
    )

    k = min(10, len(labels))
    discounts = np.log2(np.arange(2, k + 2))
    dcg = (labels[:k] / discounts).sum()

    ideal_labels = np.sort(labels)[::-1][:k]
    ideal_dcg = (ideal_labels / discounts).sum()

    ndcg_10 = dcg / ideal_dcg if ideal_dcg else 0

    return {
        "Precision@5": precision_5,
        "Precision@10": precision_10,
        "MRR": mrr,
        "AP": ap,
        "NDCG@10": ndcg_10
    }


In [78]:

bow_ranked = ranked_by_model("similarity_1_bow","clean_job_title","relevance")
bow_metrics = metrics(bow_ranked, "relevance")

tfidf_ranked = ranked_by_model("similarity_1_tfidf","clean_job_title","relevance")
tfidf_metrics = metrics(bow_ranked, "relevance")

glove_ranked = ranked_by_model("similarity_1_glove","clean_job_title","relevance")
glove_metrics = metrics(glove_ranked, "relevance")

bert_ranked = ranked_by_model("similarity_1_bert","clean_job_title","relevance")
bert_metrics = metrics(bert_ranked, "relevance")

sbert_ranked = ranked_by_model("similarity_1_sbert","clean_job_title","relevance")
sbert_metrics = metrics(sbert_ranked, "relevance")

e5_ranked = ranked_by_model("similarity_1_5e","clean_job_title","relevance")
e5_metrics = metrics(e5_ranked, "relevance")

bge_ranked = ranked_by_model("similarity_1_bge","clean_job_title","relevance")
bge_metrics = metrics(bge_ranked, "relevance")

In [79]:
bow_ranked = ranked_by_model("similarity_2_bow","clean_job_title","relevance")
bow_metrics = metrics(bow_ranked, "relevance")

tfidf_ranked = ranked_by_model("similarity_2_tfidf","clean_job_title","relevance")
tfidf_metrics = metrics(bow_ranked, "relevance")

glove_ranked = ranked_by_model("similarity_2_glove","clean_job_title","relevance")
glove_metrics = metrics(glove_ranked, "relevance")

bert_ranked = ranked_by_model("similarity_2_bert","clean_job_title","relevance")
bert_metrics = metrics(bert_ranked, "relevance")

sbert_ranked = ranked_by_model("similarity_2_sbert","clean_job_title","relevance")
sbert_metrics = metrics(sbert_ranked, "relevance")

e5_ranked = ranked_by_model("similarity_2_5e","clean_job_title","relevance")
e5_metrics = metrics(e5_ranked, "relevance")

bge_ranked = ranked_by_model("similarity_2_bge","clean_job_title","relevance")
bge_metrics = metrics(bge_ranked, "relevance")

In [85]:
results = pd.DataFrame({
    "Bow": bow_metrics,
    "TF-IDF": tfidf_metrics,
    "GloVe": glove_metrics,
    "BERT": bert_metrics,
    "SBERT": sbert_metrics,
    "E5": e5_metrics,
    "BGE": bge_metrics
}).T.rename_axis("Model").reset_index()

results

,Model,Precision@5,Precision@10,MRR,AP,NDCG@10
0,Bow,1.0,1.0,1.0,0.990602,1.000000
1,TF-IDF,1.0,1.0,1.0,0.990602,1.000000
2,GloVe,1.0,1.0,1.0,0.963456,1.000000
3,BERT,1.0,0.7,1.0,0.849221,0.778807
4,SBERT,1.0,1.0,1.0,0.996013,1.000000
5,E5,1.0,1.0,1.0,0.993129,1.000000
6,BGE,1.0,1.0,1.0,0.993129,1.000000


In [ ]:
star_candidate = X_e5[6] # student humber college aspiring human resource

starred_similarity = cosine_similarity(
    X_e5,
    star_candidate.reshape(1, -1)
).flatten()

alpha = 0.6

df['rerank_score'] = (
    alpha * score_2.flatten()
    + (1 - alpha) * starred_similarity
)

df[["all_data", "rerank_score"]].sort_values(
    "rerank_score",
    ascending=False
).head(10)

,all_data,rerank_score
27,"Seeking Human Resources Opportunities Chicago, Illinois 390",0.825121
6,Student at Humber College and Aspiring Human Resources Generalist Kanada 61,0.801664
72,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0.799745
9,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,0.799514
2,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",0.790488
98,"Seeking Human Resources Position Las Vegas, Nevada Area 48",0.785686
5,Aspiring Human Resources Specialist Greater New York City Area 1,0.773650
93,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,0.771461
26,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",0.770525
75,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212",0.767285


In [ ]:
star_candidates = X_e5[2:6] # student number college aspiring human resource
star_anchor = star_candidates.mean(axis=0)
starred_similarity = cosine_similarity(
    X_e5,
    star_anchor.reshape(1, -1)
).flatten()

alpha = 0.5

df['rerank_scores'] = (
    alpha * score_2.flatten()
    + (1 - alpha) * starred_similarity
)

df[["job_title", "rerank_scores"]].sort_values(
    "rerank_scores",
    ascending=False
).head(10)

,job_title,rerank_scores
27,Seeking Human Resources Opportunities,0.851140
2,Aspiring Human Resources Professional,0.848382
9,Seeking Human Resources HRIS and Generalist Positions,0.836446
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.829493
5,Aspiring Human Resources Specialist,0.828090
98,Seeking Human Resources Position,0.824765
75,Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment,0.811959
26,Aspiring Human Resources Management student seeking an internship,0.810844
93,Seeking Human Resources Opportunities. Open to travel and relocation.,0.808721
73,Human Resources Professional,0.804489


: 

In [ ]:

from transformers import pipeline

zero_shot_classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli"
)

text = "is this candidate HR related, yes or no."

candidate_labels = [
    "Yes",
    "No",
]

result = zero_shot_classifier(
    text,
    candidate_labels=candidate_labels
)

print(result)


In [ ]:
def zero_shot_classifer_check():
    zero_shot_classifier = pipeline(
        task="zero-shot-classification",
        model="facebook/bart-large-mnli"
    )

    candidate_labels = [
        "This candidate HR related",
        "This candidate not HR related"
    ]

    texts = df["all_data"].tolist()

    results = zero_shot_classifier(
        texts,
        candidate_labels=candidate_labels
    )

    return results

In [ ]:
results = zero_shot_classifer_check()
df_results = pd.DataFrame(results)
df_results

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

,sequence,labels,scores
0,"2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional Houston, Texas 85","[This candidate HR related, This candidate not HR related]","[0.9924140572547913, 0.007585996761918068]"
1,Native English Teacher at EPIK (English Program in Korea) Kanada 500,"[This candidate not HR related, This candidate HR related]","[0.5356791615486145, 0.4643208086490631]"
2,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","[This candidate HR related, This candidate not HR related]","[0.9939702749252319, 0.0060297283343970776]"
3,"People Development Coordinator at Ryan Denton, Texas 500","[This candidate HR related, This candidate not HR related]","[0.9079433679580688, 0.09205660969018936]"
4,"Advisory Board Member at Celal Bayar University Izmir, Turkiye 500","[This candidate HR related, This candidate not HR related]","[0.7797142863273621, 0.22028574347496033]"
5,Aspiring Human Resources Specialist Greater New York City Area 1,"[This candidate HR related, This candidate not HR related]","[0.9948657155036926, 0.005134335719048977]"
6,Student at Humber College and Aspiring Human Resources Generalist Kanada 61,"[This candidate HR related, This candidate not HR related]","[0.9923202991485596, 0.0076797460205852985]"
7,HR Senior Specialist San Francisco Bay Area 500,"[This candidate HR related, This candidate not HR related]","[0.9229919910430908, 0.0770079717040062]"
8,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"[This candidate HR related, This candidate not HR related]","[0.9908297061920166, 0.009170301258563995]"
9,"Student at Chapman University Lake Forest, California 2","[This candidate not HR related, This candidate HR related]","[0.6715776920318604, 0.32842233777046204]"


In [ ]:
comparison_1

,BOW,TF-IDF,GloVe,BERT,SBERT,E5,BGE
1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44"
2,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7"
3,Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212"
4,Student at Humber College and Aspiring Human Resources Generalist Kanada 61,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","Human Resources professional for the world leader in GIS software Highland, California 50","Human Resources, Staffing and Recruiting Professional Jackson, Mississippi Area 500","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",Aspiring Human Resources Specialist Greater New York City Area 1
5,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","Business Management Major and Aspiring Human Resources Manager Monroe, Louisiana Area 5",Director Human Resources at EY Greater Atlanta Area 349,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Seeking Human Resources Position Las Vegas, Nevada Area 48","Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500"
6,"Business Management Major and Aspiring Human Resources Manager Monroe, Louisiana Area 5",Student at Humber College and Aspiring Human Resources Generalist Kanada 61,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Experienced Retail Manager and aspiring Human Resources Professional Austin, Texas Area 57","Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174","Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103"
7,"Aspiring Human Re

In [ ]:
comparison_2

,BOW,TF-IDF,GloVe,BERT,SBERT,E5,BGE,relevance
1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Position Las Vegas, Nevada Area 48","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Opportunities Chicago, Illinois 390",0
2,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Human Resources Professional Greater Boston Area 16,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","Seeking Human Resources Position Las Vegas, Nevada Area 48",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,0
3,"Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Human Resources, Staffing and Recruiting Professional Jackson, Mississippi Area 500",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0
4,"Human Resources Management Major Milpitas, California 18",Aspiring Human Resources Specialist Greater New York City Area 1,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Seeking Human Resources Opportunities Chicago, Illinois 390",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Seeking Human Resources Position Las Vegas, Nevada Area 48",0
5,Human Resources Professional Greater Boston Area 16,"Human Resources Management Major Milpitas, California 18",Director Human Resources at EY Greater Atlanta Area 349,"Seeking Human Resources Position Las Vegas, Nevada Area 48",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",0
6,Human Resources Generalist at Schwan's Amerika Birlesik Devletleri 500,Director Human Resources at EY Greater Atlanta Area 349,"Human Resources Management Major Milpitas, California 18","Seeking employment opportunities within Customer Service or Patient Care Torrance, California 64","Human Resources Generalist at Loparex Raleigh-Durham, North Carolina Area 500",Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,0
7,Director Human Resources at EY Greater Atlanta Area 349,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",Human Resources Professional Greater Boston Area 16,"Director Of Administration at Excellence Logging Katy, Texas 500","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Aspiring Human Resources Specialist Greater New York City Area 1,Aspiring Human Resources Specialist Greater New York City A